In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import csv
import numpy as np

## **80% y 20%**

In [ ]:
import random

path = "/content/Datacorpus_SIN_normalizar.csv"

ruta_archivo_entrada = path
ruta_archivo_salida_entrenamiento = 'Entrenamiento.csv'
ruta_archivo_salida_prueba = 'Prueba.csv'

# Lista para almacenar los títulos + contenidos y las secciones
titulos_contenidos = []
secciones = []

with open(ruta_archivo_entrada, 'r', newline='', encoding='utf-8') as archivo_csv:
    lector_csv = csv.reader(archivo_csv)
    next(lector_csv)

    for fila in lector_csv:
        titulo_contenido = f"{fila[1]}, {fila[2]}"
        seccion = fila[3]

        titulos_contenidos.append(titulo_contenido)
        secciones.append(seccion)

# Combinar los datos en una lista de tuplas
datos_combinados = list(zip(titulos_contenidos, secciones))

# Mezclar los datos de manera aleatoria
random.shuffle(datos_combinados)

# Calcular el índice de división
indice_division = int(len(datos_combinados) * 0.8)

# Dividir los datos en entrenamiento y prueba
datos_entrenamiento = datos_combinados[:indice_division]
datos_prueba = datos_combinados[indice_division:]

with open(ruta_archivo_salida_entrenamiento, 'w', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.writer(archivo_csv)
    # Escribir la fila de encabezados
    escritor_csv.writerow(['Titulo + contenidos', 'section'])

    # Escribir las filas de datos de entrenamiento
    for fila in datos_entrenamiento:
        escritor_csv.writerow(fila)

with open(ruta_archivo_salida_prueba, 'w', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.writer(archivo_csv)
    escritor_csv.writerow(['Titulo + contenidos', 'section'])

    # Escribir las filas de datos de prueba
    for fila in datos_prueba:
        escritor_csv.writerow(fila)

print(f"Archivos '{ruta_archivo_salida_entrenamiento}' y '{ruta_archivo_salida_prueba}' creados exitosamente.")


Archivos 'Entrenamiento.csv' y 'Prueba.csv' creados exitosamente.


## **Precesamiento de texto**

## ***Normalizacion con spacy***

In [ ]:
import re
import csv
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

# Modelo de spaCy para español
nlp = spacy.load('es_core_news_sm')

def tokenizacion(texto):
    return word_tokenize(texto.lower())

def text_cleaning(texto):
    texto = re.sub(r'\W', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.strip()
    return texto

def eliminar_stop_words(tokens):
    stop_words = set(stopwords.words('spanish'))
    return [word for word in tokens if word not in stop_words]

def lematizacion(texto):
    doc = nlp(texto)
    return [token.lemma_ for token in doc]

def procesar_texto(opcion, texto):
    if opcion == 'A':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        return ' '.join(tokens)  # Text cleaning y tokenización
    elif opcion == 'B':
        tokens = tokenizacion(texto)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        return ' '.join(tokens_sin_stopwords)  # Tokenización y stop words
    elif opcion == 'C':
        tokens_sin_stopwords = eliminar_stop_words(tokenizacion(texto))
        tokens_lematizados = lematizacion(' '.join(tokens_sin_stopwords))
        return ' '.join(tokens_lematizados)  # Stop words y lematización
    elif opcion == 'D':
        texto_limpio = text_cleaning(texto)
        tokens_lematizados = lematizacion(texto_limpio)
        return ' '.join(tokens_lematizados)  # Text cleaning y lematización
    elif opcion == 'E':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        return ' '.join(tokens_sin_stopwords)  # Text cleaning, tokenización y stop words
    elif opcion == 'F':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_lematizados = lematizacion(' '.join(tokens))
        return ' '.join(tokens_lematizados)  # Text cleaning, tokenización y lematización
    elif opcion == 'G':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        tokens_lematizados = lematizacion(' '.join(tokens_sin_stopwords))
        return ' '.join(tokens_lematizados)  # Text cleaning, tokenización, stop words y lematización

def leer_y_procesar_csv(ruta_archivo, opcion):
    titulos_contenidos = []
    secciones = []

    with open(ruta_archivo, 'r', newline='', encoding='utf-8') as archivo_csv:
        lector_csv = csv.reader(archivo_csv)
        next(lector_csv)  # Saltar la primera fila (títulos)

        for fila in lector_csv:
            titulo_contenido = fila[0]
            seccion = fila[1]
            texto_procesado = procesar_texto(opcion, str(titulo_contenido))

            titulos_contenidos.append(texto_procesado)
            secciones.append(seccion)

    return titulos_contenidos, secciones

def escribir_csv(titulos_contenidos, secciones, nombre_archivo):
    with open(nombre_archivo, 'w', newline='', encoding='utf-8') as archivo_csv:
        escritor_csv = csv.writer(archivo_csv)
        escritor_csv.writerow(['Titulo + contenidos', 'section'])

        for i in range(len(titulos_contenidos)):
            escritor_csv.writerow([titulos_contenidos[i], secciones[i]])

ruta_archivo_entrenamiento = 'Entrenamiento.csv'
ruta_archivo_prueba = 'Prueba.csv'

def procesar_opciones(ruta_archivo):
    opciones = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    for opcion in opciones:
        nombre_archivo_salida = f'{ruta_archivo.split(".")[0]}_{opcion}.csv'
        titulos_contenidos, secciones = leer_y_procesar_csv(ruta_archivo, opcion)
        escribir_csv(titulos_contenidos, secciones, nombre_archivo_salida)
        print(f"Archivo '{nombre_archivo_salida}' creado exitosamente.")

# Procesar Entrenamiento y Prueba con todas las opciones
procesar_opciones(ruta_archivo_entrenamiento)
procesar_opciones(ruta_archivo_prueba)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Archivo 'Entrenamiento_A.csv' creado exitosamente.
Archivo 'Entrenamiento_B.csv' creado exitosamente.
Archivo 'Entrenamiento_C.csv' creado exitosamente.
Archivo 'Entrenamiento_D.csv' creado exitosamente.
Archivo 'Entrenamiento_E.csv' creado exitosamente.
Archivo 'Entrenamiento_F.csv' creado exitosamente.
Archivo 'Entrenamiento_G.csv' creado exitosamente.
Archivo 'Prueba_A.csv' creado exitosamente.
Archivo 'Prueba_B.csv' creado exitosamente.
Archivo 'Prueba_C.csv' creado exitosamente.
Archivo 'Prueba_D.csv' creado exitosamente.
Archivo 'Prueba_E.csv' creado exitosamente.
Archivo 'Prueba_F.csv' creado exitosamente.
Archivo 'Prueba_G.csv' creado exitosamente.
